In [12]:
import shutil
import os

def crea_copia_gt(path_gt, suffix="_tmp"):
    """
    Crea una copia del file gt.txt e restituisce il path della copia.
    """
    base, ext = os.path.splitext(path_gt)
    path_copia = f"{base}{suffix}{ext}"

    shutil.copy(path_gt, path_copia)
    return path_copia


In [13]:
def ordina_gt_per_prima_colonna(path_in, path_out=None):
    """
    Ordina le righe di un file CSV (comma separated) in base alla prima colonna.
    Se path_out è None, sovrascrive il file di input.
    """
    # Leggi tutte le righe
    with open(path_in, "r") as f:
        righe = [line.strip() for line in f if line.strip()]

    # Ordina usando la prima colonna come intero
    righe.sort(key=lambda r: int(r.split(",")[0]))

    # Scrivi il risultato
    output = path_out if path_out is not None else path_in
    with open(output, "w") as f:
        for r in righe:
            f.write(r + "\n")


In [14]:
def leggi_gt_struct(path):
    with open(path, "r") as f:
        for line in f:
            v = list(map(int, line.strip().split(",")))
            yield {
                "frame": v[0],
                "id": v[1],
                "x": v[2],
                "y": v[3],
                "w": v[4],
                "h": v[5],
                "conf": v[6],
                "cls": v[7],
                "vis": v[8],
                "extra": v[9] if len(v) > 9 else None
            }


In [15]:
import json

def load_rois_absolute(json_path, img_width, img_height):
    """
    Legge un JSON con ROI in coordinate relative e le converte in coordinate assolute.

    :param json_path: path al file JSON
    :param img_width: larghezza dell'immagine (pixel)
    :param img_height: altezza dell'immagine (pixel)
    :return: lista di tuple (x, y, width, height) in coordinate assolute
    """
    with open(json_path, "r") as f:
        rois = json.load(f)

    absolute_rois = []

    for roi_name, roi in rois.items():
        x_abs = roi["x"] * img_width
        y_abs = roi["y"] * img_height
        width_abs = roi["width"] * img_width
        height_abs = roi["height"] * img_height

        absolute_rois.append((x_abs, y_abs, width_abs, height_abs))

    return absolute_rois

def is_bbox_base_center_inside_roi(roi, x1_i, y1_i, bw_i, bh_i):
    """
    Verifica se il centro della base di un bounding box è dentro una ROI.

    :param roi: tupla (x_roi, y_roi, w_roi, h_roi) in coordinate assolute
    :param x1_i: top-left x del bounding box
    :param y1_i: top-left y del bounding box
    :param bw_i: larghezza del bounding box
    :param bh_i: altezza del bounding box
    :return: True se il centro della base è nella ROI, False altrimenti
    """
    x_roi, y_roi, w_roi, h_roi = roi

    # centro della base del bounding box
    cx = x1_i + bw_i / 2
    cy = y1_i + bh_i

    # limiti ROI
    inside_x = x_roi <= cx <= (x_roi + w_roi)
    inside_y = y_roi <= cy <= (y_roi + h_roi)

    return inside_x and inside_y

def append_rois_results_to_txt(rois_results, file_path):
    """
    Scrive i risultati delle ROI su file TXT, una riga per tupla, comma-separated.
    Se il file esiste, i nuovi dati vengono aggiunti in coda.

    :param rois_results: lista di tuple (frame_id, roi_index, count)
    :param file_path: path del file txt
    """
    with open(file_path, "a") as f:
        for result in rois_results:
            line = ",".join(str(v) for v in result)
            f.write(line + "\n")

In [16]:
import json
import random
import os

def genera_roi_json(output_path: str):
    """
    Genera un file roi.json con valori casuali tra 0 e 1
    nel percorso specificato.
    """

    roi_data = {
        "roi1": {
            "x": random.random(),
            "y": random.random(),
            "width": random.random(),
            "height": random.random()
        },
        "roi2": {
            "x": random.random(),
            "y": random.random(),
            "width": random.random(),
            "height": random.random()
        }
    }

    # Se viene passato un percorso di directory, aggiunge roi.json
    if os.path.isdir(output_path):
        file_path = os.path.join(output_path, "roi.json")
    else:
        file_path = output_path

    with open(file_path, "w") as f:
        json.dump(roi_data, f, indent=4)

    return file_path

In [ ]:
import configparser
import re

def get_ball_tracklet_id(ini_path):
    config = configparser.ConfigParser()
    config.read(ini_path)

    for key, value in config["Sequence"].items():
        # cerco righe tipo: trackletID_18 = ball;1
        if value.strip().startswith("ball;"):
            match = re.search(r"trackletid_(\d+)", key)
            if match:
                return match.group(1)

    return None

In [ ]:
import csv
import os
import tempfile

def rimuovi_righe_palla(src_file, ini_file_path):
    """
    Rimuove dal file CSV le righe di ground truth relative alla palla,
    identificata tramite get_ball_tracklet_id.

    :param src_file: percorso del file CSV da modificare
    :param ini_file_path: percorso del file ini usato per ottenere l'ID della palla
    """
    ball_tracklet_id = get_ball_tracklet_id(ini_file_path)

    temp_file = tempfile.NamedTemporaryFile(
        mode='w',
        newline='',
        delete=False,
        encoding='utf-8'
    )

    with open(src_file, newline='', encoding='utf-8') as f, temp_file:
        reader = csv.reader(f, delimiter=',')
        writer = csv.writer(temp_file, delimiter=',')

        for row in reader:
            if row[1] != ball_tracklet_id:
                writer.writerow(row)

    os.replace(temp_file.name, src_file)


In [ ]:
base_path = "/Users/francescopiocirillo/Desktop/SIMULATOR/lecture_example_from_training/test_set/videos"

for folder in os.listdir(base_path):
    if folder.startswith('.'):
        continue
    
    folder_path = os.path.join(base_path, folder)
    roi_path = os.path.join(folder_path, "roi.json")
    gt_path = os.path.join(folder_path, "gt", "gt.txt")
    behavior_path = os.path.join(folder_path, "gt", "behavior_gt.txt")
    ini_path = os.path.join(folder_path, "gameinfo.ini")

    rimuovi_righe_palla(gt_path, ini_path)

    gt_copy = crea_copia_gt(gt_path)
    ordina_gt_per_prima_colonna(gt_copy)

    genera_roi_json(folder_path)
    rois = load_rois_absolute(roi_path, img_width=1920, img_height=1080)

    current_frame = None
    roi_counts = [0] * len(rois)
    rois_results = []
    for record in leggi_gt_struct(gt_copy):
        if current_frame is None:
            current_frame = record["frame"]
            roi_counts = [0] * len(rois)

        if current_frame != record["frame"]:
            for i, roi in enumerate(rois):
                rois_results.append((current_frame, i+1, roi_counts[i]))
            append_rois_results_to_txt(rois_results, behavior_path)
            roi_counts = [0] * len(rois)
            rois_results = []
            current_frame = record["frame"]

        for i, roi in enumerate(rois):
            if is_bbox_base_center_inside_roi(roi, record["x"], record["y"], record["w"], record["h"]):
                roi_counts[i] += 1

    for i, roi in enumerate(rois):
        rois_results.append((current_frame, i+1, roi_counts[i]))
        append_rois_results_to_txt(rois_results, behavior_path)

    os.remove(gt_copy)